# 라이브러리 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT * 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_attendance`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

c:\workspace\final_project\sns_service_analysis\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,attendance_date_list,user_id
0,7923,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",1355335
1,38921,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",892265
2,777,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",913054
3,6557,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",947077
4,2534,"[""2023-05-27"", ""2023-05-28"", ""2023-05-29"", ""20...",1233366


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 349637 entries, 0 to 349636
Data columns (total 3 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   id                    349637 non-null  Int64
 1   attendance_date_list  349637 non-null  str  
 2   user_id               349637 non-null  Int64
dtypes: Int64(2), str(1)
memory usage: 38.4 MB


## 데이터 타입 등 정보 확인
- 결측값 확인 안됨
- attendance_date_list str 타입.

In [5]:
print(df.duplicated().sum())
df['user_id'].duplicated().sum()

0


np.int64(0)

## 중복값 확인
- 전체 중복값 0
- 유저아이디 기준 중복값 0

# 특이값 확인

In [6]:
df[df['attendance_date_list'] == '[]']

,id,attendance_date_list,user_id
328692,20,[],1484400
328693,243,[],1284925
328694,312,[],1451267
328695,339,[],1483247
328696,375,[],1173480
...,...,...,...
349632,360487,[],1162275
349633,360491,[],935331
349634,360497,[],1127910
349635,360498,[],851491


In [7]:
# str형태의 데이터를 리스트로 변환

df['attendance_date_list'] = (
    df['attendance_date_list']
    .apply(ast.literal_eval)
)

In [8]:
# 리스트형식 날짜형으로 변환 후 첫날 날짜와 마지막 날짜 확인 

dates = pd.to_datetime(
    df['attendance_date_list'].explode(),
    errors='coerce'
)

dates.agg(['min', 'max'])

min   2023-05-27
max   2024-05-09
Name: attendance_date_list, dtype: datetime64[us]

In [9]:
# 가입일자 확인용 유저 테이블 호출
sql = f"""
    SELECT id,
        created_at 
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user`
"""

# 판다스 데이터프레임으로 변환
df1 = client.query(sql).to_dataframe()

df1.head()

c:\workspace\final_project\sns_service_analysis\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,created_at
0,831956,2023-03-29 03:44:14.047130+00:00
1,831962,2023-03-29 05:18:56.162368+00:00
2,832151,2023-03-29 12:56:34.989468+00:00
3,832340,2023-03-29 12:56:35.020790+00:00
4,832520,2023-03-29 12:56:35.049311+00:00


In [10]:
# 출석 테이블에 존재하는 유저를 변수에 지정 > 유저 테이블에서 해당 유저 필터링

check_id = df['user_id']

check_df = df1[~(df1['id'].isin(check_id))]
print(f"출석 테이블에 존재하지 않는 유저 : {check_df.shape[0]}명")
print(f"출석 테이블에 존재하지 않는 유저의 가장 빠른 가입일 : {check_df['created_at'].min()}")
print(f"출석 테이블에 존재하지 않는 유저의 가장 늦은 가입일 : {check_df['created_at'].max()}")

# 빈 리스트의 값을 가진 유저를 변수에 지정 > 유저 테이블에서 해당 유저 필터링

check_id1 = df[df['attendance_date_list'].apply(len) == 0]['user_id']

check_df1 = df1[df1['id'].isin(check_id1)]
print(f"출석 테이블의 빈 리스트 값을 가진 유저 : {check_df1.shape[0]}명")
print(f"빈 리스트 값을 가진 유저의 가장 빠른 가입일 : {check_df1['created_at'].min()}")
print(f"빈 리스트 값을 가진 유저의 가장 늦은 가입일 :{check_df1['created_at'].max()}")


출석 테이블에 존재하지 않는 유저 : 327448명
출석 테이블에 존재하지 않는 유저의 가장 빠른 가입일 : 2023-03-29 03:44:14.047130+00:00
출석 테이블에 존재하지 않는 유저의 가장 늦은 가입일 : 2024-05-09 08:31:17.710824+00:00
출석 테이블의 빈 리스트 값을 가진 유저 : 20945명
빈 리스트 값을 가진 유저의 가장 빠른 가입일 : 2023-03-31 16:55:28.590111+00:00
빈 리스트 값을 가진 유저의 가장 늦은 가입일 :2024-05-05 04:48:17.223190+00:00


### 출석 테이블 빈 리스트 값 확인 결과
- 출석 테이블의 349,637명 중 20,945명(약 6%)에서 attendance_date_list가 빈 리스트([])로 확인되었다.
- 빈 리스트 발생 원인을 확인하기 위해 와이어프레임과 출석 날짜 범위를 확인한 결과, 최초 출석 기록은 2023-05-27부터 존재하여 해당 시점 전후에 출석 기능이 도입된 것으로 추정하였다.
- 기능 도입 이후 가입자에게 출석 리스트가 자동 생성되는 구조인지 확인하였으나, 출석 테이블에 레코드가 없는 유저와 빈 리스트를 가진 유저의 가입 시점이 서로 겹치는 것으로 확인되었다. 따라서 단순히 기능 도입 이후 가입으로 인해 빈 리스트가 생성되는 구조라고 보기는 어려웠다.
- 추가로 와이어프레임을 확인한 결과, 출석 데이터는 홈 화면의 출석체크 버튼 상호작용을 통해 수집되는 데이터로 판단하였다. 따라서 해당 테이블은 전체 서비스 이용 로그를 나타내는 데이터가 아니며, 이 테이블만으로 서비스 전체의 활성일자·유지기간·리텐션을 측정하기에는 한계가 있다.
- 최종적으로 분석 목적상 실제 출석 이벤트가 존재하지 않는 빈 리스트 레코드는 활용 가치가 낮다고 판단하여, 349,637명 중 20,945명(약 6%)의 빈 리스트 레코드를 분석 대상에서 제외하는 것으로 전처리하였다.

## 유저별 출석 날짜 중복 및 이상값 체크

In [11]:
# 리스트 내 데이터 중복 여부 및 갯수

df['has_duplicate'] = df['attendance_date_list'].apply(
    lambda x: len(x) != len(set(x))
)
# 중복이 있었다면 True, 없다면 False로 값 입력

df[df['has_duplicate']]

df['has_duplicate'].sum()

np.int64(0)

In [12]:
# 리스트의 값을 하나씩 형태나 비정상 데이터를 판단.

def find_invalid_dates(date_list):
    invalid = []

    for date in date_list:
        parsed = pd.to_datetime(
            date,
            format='%Y-%m-%d',
            errors='coerce'
        )

        if pd.isna(parsed):
            invalid.append(date)

    return invalid

In [13]:
df['invalid_dates'] = (
    df['attendance_date_list']
    .apply(find_invalid_dates)
)

In [14]:
(df['invalid_dates'].apply(len) > 0).sum()

np.int64(0)

- 리스트 안의 날짜는 모두 이상이 없는 것으로 확인되었으며, 중복도 존재하지 않기 때문에 다른 처리 없이 전처리를 마무리 함.

In [ ]:

# 출석 테이블 빈 리스트 데이터 삭제


# 삭제 대상 사전 확인
check_sql = f"""
SELECT
    COUNT(*) AS delete_cnt
FROM `{PROJECT_ID}.{DATA_SET}.accounts_attendance`
WHERE attendance_date_list = '[]'
"""

check_df = client.query(check_sql).to_dataframe()

delete_cnt = check_df.loc[0, 'delete_cnt']

print(f"삭제 예정 행 수: {delete_cnt:,}")

# 실제 삭제

if delete_cnt == 20945:

    delete_sql = f"""
    DELETE FROM `{PROJECT_ID}.{DATA_SET}.accounts_attendance`
    WHERE attendance_date_list = '[]'
    """

    query_job = client.query(delete_sql)
    query_job.result()

    print(
        f"삭제 완료: "
        f"{query_job.num_dml_affected_rows:,}행"
    )

else:
    print(
        f"[경고] 예상한 20,945행과 다릅니다. "
        f"현재 대상: {delete_cnt:,}행\n"
        "삭제를 중단합니다."
    )

# 삭제 후 검증

verify_sql = f"""
SELECT
    COUNT(*) AS remaining_cnt
FROM `{PROJECT_ID}.{DATA_SET}.accounts_attendance`
WHERE attendance_date_list = '[]'
"""

verify_df = client.query(verify_sql).to_dataframe()

print(
    "삭제 후 남은 빈 리스트:",
    verify_df.loc[0, 'remaining_cnt']
)